# Immigration Guidance RAG with Claude

This notebook implements a Retrieval-Augmented Generation (RAG) system for USCIS immigration policy guidance using:
- **LLM**: Claude Sonnet 4.5 (via Anthropic API)
- **Embeddings**: HuggingFace sentence-transformers/all-MiniLM-L6-v2
- **Vectorstore**: Chroma
- **Framework**: LangChain

## 1. Setup and Dependencies

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv


load_dotenv()

if not os.getenv("ANTHROPIC_API_KEY"):
    print("⚠️  Warning: ANTHROPIC_API_KEY not found in environment variables")
    print("Please set it in a .env file or directly in this cell")
else:
    print("✓ API key loaded successfully")

✓ API key loaded successfully


## 2. Document Loading

Load all PDF documents from the data/raw directory.

In [2]:
from langchain_community.document_loaders import PyPDFLoader


pdf_dir = Path(r"A:\rag-immigration-guidance\data\raw")
pdf_files = list(pdf_dir.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f"  - {pdf.name}")


documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    documents.extend(loader.load())

print(f"\n✓ Total pages loaded: {len(documents)}")

a:\rag-immigration-guidance\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 47 PDF files:
  - Application for Employment Authorization _ USCIS.pdf
  - Application to Register Permanent Residence or Adjust Status _ USCIS.pdf
  - Chapter 1 - Purpose and Background _ USCIS.pdf
  - Chapter 1 - Purpose and Background.pdf
  - Chapter 10 - Decision and Post-Adjudication _ USCIS.pdf
  - Chapter 10 - Legal Analysis and Use of Discretion _ USCIS.pdf
  - Chapter 2 - Accommodation Policies and Procedures _ USCIS.pdf
  - Chapter 2 - Application for Replacement_Initial Nonimmigrant Arrival-Departure Record (Form I-102) _ USCIS.pdf
  - Chapter 2 - Background and Security Checks _ USCIS.pdf
  - Chapter 2 - Becoming a U.S. Citizen _ USCIS.pdf
  - Chapter 2 - Eligibility Requirements 2_ USCIS.pdf
  - Chapter 2 - Eligibility Requirements _ USCIS.pdf
  - Chapter 2 - J Exchange Visitor Eligibility _ USCIS.pdf
  - Chapter 2 - Replacement of Permanent Resident Card _ USCIS.pdf
  - Chapter 2 - USCIS-Issued Secure Identity Documents _ USCIS.pdf
  - Chapter 3 - Expired Permanent 

## 3. Text Chunking

Split documents into smaller chunks for better retrieval precision.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)


split_docs = text_splitter.split_documents(documents)

print(f"✓ Total chunks created: {len(split_docs)}")
print(f"\nExample chunk:")
print(f"Text: {split_docs[0].page_content[:200]}...")
print(f"Metadata: {split_docs[0].metadata}")

✓ Total chunks created: 913

Example chunk:
Text: Home>  Forms>  All Forms>  Application for Employment Authorization
I-765, Application for EmploymentAuthorization
Alert: On Jan. 9, 2026, DHS announced the Adjustment to Premium Processing Fees fina...
Metadata: {'producer': 'Skia/PDF m144', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36', 'creationdate': '2026-02-01T00:37:38+00:00', 'title': 'Application for Employment Authorization | USCIS', 'moddate': '2026-02-01T00:37:38+00:00', 'source': 'A:\\rag-immigration-guidance\\data\\raw\\Application for Employment Authorization _ USCIS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## 4. Create Embeddings and Vectorstore

Generate embeddings using HuggingFace and store them in Chroma vectorstore.

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("Creating vectorstore... (this may take a minute)")


vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_db" 
)

print(f"✓ Vectorstore created with {vectorstore._collection.count()} chunks")

C:\Users\aaron fraser\AppData\Local\Temp\ipykernel_1628\165916275.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2179.35it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating vectorstore... (this may take a minute)
✓ Vectorstore created with 5812 chunks


## 5. Set Up Claude with LangChain

Initialize Claude Sonnet 4.5 as the LLM for generation.

In [5]:
from langchain_anthropic import ChatAnthropic


llm = ChatAnthropic(
    model="claude-sonnet-4-5-20250929",
    temperature=0,
    max_tokens=4096
)

print("✓ Claude Sonnet 4.5 initialized")

✓ Claude Sonnet 4.5 initialized


## 6. Create RAG Chain

Build the complete RAG pipeline with retrieval and generation.

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


prompt_template = """You are an expert immigration law assistant with deep knowledge of USCIS policies and procedures. 
Use the following context from official USCIS documentation to answer the question accurately and comprehensively.

IMPORTANT: 
- Cite specific sources by referencing the document chapter and page numbers
- If the context doesn't contain enough information to fully answer the question, acknowledge this
- Provide clear, actionable guidance when applicable
- Use precise legal terminology from the source documents

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | PROMPT
    | llm
    | StrOutputParser()
)

print("✓ RAG chain created successfully using LCEL")

✓ RAG chain created successfully using LCEL


## 7. Query the RAG System

Test the system with immigration-related questions.

In [7]:
def ask_question(question: str):
    """
    Query the RAG system and display results with sources.
    """
    print(f"\n{'='*80}")
    print(f"QUESTION: {question}")
    print(f"{'='*80}\n")
    
   
    source_docs = retriever.invoke(question)
    
 
    answer = rag_chain.invoke(question)


    print("ANSWER:")
    print(answer)
    
    
    print(f"\n{'─'*80}")
    print("SOURCES:")
    for i, doc in enumerate(source_docs, 1):
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "Unknown")
        print(f"  [{i}] {Path(source).name} - Page {page}")
    print(f"{'='*80}\n")
    
    return {"result": answer, "source_documents": source_docs}

In [8]:
s
result1 = ask_question("What are the key factors and requirements to gain U.S. citizenship?")

NameError: name 's' is not defined

In [ ]:

result2 = ask_question("What documentation is required for an immigration application?")


QUESTION: What documentation is required for an immigration application?

ANSWER:
Based on the provided USCIS documentation, I can provide information about documentation requirements, though the context focuses primarily on naturalization applications and A-files rather than comprehensive immigration application requirements.

## Documentation for Naturalization Applications

According to **Chapter 4 - Documentation** (specific section on "Preliminary Review of Application"), the following documentation is reviewed:

### A-File Contents
The applicant's **A-file** (Alien file) contains the applicant's record of interaction with USCIS and legacy INS, and may include:

- **Documents showing how the applicant became an LPR** (Lawful Permanent Resident)
- **Other applications or forms for immigration benefits** submitted by the applicant
- **Correspondence between USCIS and the applicant**
- **Memoranda and forms from officers** that may be pertinent to the applicant's eligibility

### Ad

In [ ]:

result3 = ask_question("What should first time applicants expect?")


QUESTION: What should first time applicants expect?

ANSWER:
Based on the provided USCIS documentation, I can provide information about what applicants should expect during the naturalization interview process, though the context appears to focus primarily on re-examinations rather than first-time initial interviews.

## What First-Time Applicants Should Expect:

### Interview Topics and Questions

According to the documentation, the officer will ask questions covering multiple areas, including but not limited to:

- **Biographical information**, including marital history and military service
- **Admission and length of time as a lawful permanent resident (LPR)**
- **Absences from the United States** after becoming an LPR
- **Places of residence and employment history**
- **Knowledge of English and of U.S. history and government (civics)**
- **Moral character and any criminal history**
- **Attachment to the principles of the U.S. Constitution**
- **Affiliations or memberships in certa

In [ ]:
result4 = ask_question("what are the requirments for citizenship")


QUESTION: what are the requirments for citizenship

ANSWER:
Based on the provided USCIS documentation, I can provide some information about citizenship requirements, though the context provided is limited.

## General Requirements Mentioned:

According to **Chapter 2 - Becoming a U.S. Citizen (Volume 12, Part A, Chapter 2, Page 1/3)**:

1. **Lawful Permanent Residence**: In most cases, a person may not be naturalized unless he or she has been lawfully admitted to the United States for permanent residence.

2. **Congressional Requirements**: The applicant must fulfill all of the requirements established by Congress.

## Pathways to Citizenship:

The document identifies two main pathways:

1. **Naturalization through application** - The standard process where an individual applies for citizenship
2. **Derivation of citizenship** - Naturalization by operation of law (often referred to as "deriving citizenship")

## Important Limitation:

**The provided context does NOT contain the comple

## 8. Advanced: Custom Query Function

For more control over retrieval and formatting.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

def ask_with_custom_retrieval(question: str, num_chunks: int = 4):
    """
    Custom RAG implementation with manual retrieval and formatting.
    """
    
    docs = vectorstore.similarity_search(question, k=num_chunks)
    
    
    context_parts = []
    for i, doc in enumerate(docs, 1):
        source = Path(doc.metadata.get("source", "Unknown")).name
        page = doc.metadata.get("page", "Unknown")
        context_parts.append(f"[Source {i}: {source}, Page {page}]\n{doc.page_content}")
    
    context = "\n\n".join(context_parts)
    
    
    messages = [
        SystemMessage(content="You are an expert immigration law assistant specializing in USCIS policies."),
        HumanMessage(content=f"""Based on the following context from USCIS documentation, answer the question accurately and cite your sources.

Context:
{context}

Question: {question}

Provide a comprehensive answer with specific citations to the source documents.""")
    ]
    
    
    response = llm.invoke(messages)
    
    return {
        "answer": response.content,
        "sources": [(Path(doc.metadata["source"]).name, doc.metadata["page"]) for doc in docs]
    }

In [ ]:

custom_result = ask_with_custom_retrieval(
    "What should a first time applicant expect?",
    num_chunks=6
)

print("ANSWER:")
print(custom_result["answer"])
print("\nSOURCES:")
for source, page in custom_result["sources"]:
    print(f"  - {source}, Page {page}")

ANSWER:
# What a First-Time Naturalization Applicant Should Expect

Based on the USCIS documentation provided, a first-time naturalization applicant should expect the following during their naturalization interview:

## Interview Topics and Questions

During the naturalization interview, the USCIS officer will ask questions covering various aspects of eligibility. According to the documentation, "the officer's questions may include, but are not limited to, the following questions:

- Biographical information, to include marital history and military service;
- Admission and length of time as a lawful permanent resident (LPR);
- Absences from the United States after becoming an LPR;
- Places of residence and employment history;
- Knowledge of English and of U.S. history and government (civics);
- Moral character and any criminal history;
- Attachment to the principles of the U.S. Constitution;
- Affiliations or memberships in certain organizations;
- Willingness to take an Oath of Allegi

## 10. Tips for Optimization

### Improve Retrieval Quality:
- Adjust `chunk_size` and `chunk_overlap` in text splitting
- Increase/decrease `k` (number of retrieved chunks)
- Try different retrieval methods: `similarity`, `mmr` (maximal marginal relevance)

### Improve Answer Quality:
- Modify the prompt template for more specific guidance
- Adjust Claude's `temperature` (0 = deterministic, 1 = creative)
- Use Claude Opus 4.5 for more complex legal reasoning

### Cost Optimization:
- Use Claude Haiku 4 for simpler queries
- Reduce `max_tokens` if answers are too long
- Cache frequently asked questions